In [2]:
import torch
import torch.nn as nn
from einops import rearrange

In [9]:
LLM_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_prj = nn.Linear(d_out, d_out)
        self.drop = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x):
        _, l, _ = x.shape
        q = rearrange(self.W_query(x), "b l (h k) -> b h l k", h = self.num_heads)
        k = rearrange(self.W_key(x), "b t (h k) -> b h t k", h = self.num_heads)
        v = rearrange(self.W_value(x), "b t (h v) -> b h t v", h = self.num_heads)

        attn_scores = torch.einsum("bhlk, bhtk -> bhlt", [q,k]) / k.shape[-1]**0.5
        mask_bool = self.mask.bool()[:l, :l]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_scores = torch.softmax(attn_scores, dim=-1)
        attn_scores = self.drop(attn_scores)
        context_vec = torch.einsum("bhlt, bhtv -> bhlv", [attn_scores, v])
        context_vec = self.out_prj(rearrange(context_vec, "b h l v -> b l (h v)"))

        return context_vec




In [8]:
torch.manual_seed(96)
mha = MultiHeadAttention(d_in=4, d_out=12, context_length=6, dropout=0.5, num_heads=4)
batch = torch.randn(2, 6, 4)
out = mha(batch)
print(out.shape)

torch.Size([2, 6, 12])


In [ ]:
class Transformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.mha = MultiHeadAttention(
            d_in = config["emb_dim"], d_out=config["emb_dim"],
            context_length=config["context_length"], dropout=config["drop_rate"],
            num_heads=config["n_heads"], qkv_bias=config["qkv_bias"]
        )
        self.ff = nn.Sequential(
            nn.Linear(config["emb_dim"], 4 * config["emb_dim"]),
            nn.SiLU(),
            nn.Linear(4 * config["emb_dim"], config["emb_dim"]),
        )
        self.norm1 = nn.RMSNorm(config["emb_dim"])
        self.norm2 = nn.RMSNorm(config["emb_dim"])
        self.drop = nn.Dropout(config["drop_rate"])
        
    def forward(self, x):
        sc = x
        x = self.mha(x)
        x = self.norm1(x)
        x += sc

        sc = x
        x = self.ff(x)
        x = self.norm2(x)
        x += sc
        return x

In [11]:
torch.manual_seed(69)
batch = torch.randn(2, 4, 768)
bl = Transformer(LLM_CONFIG)
out = bl(batch)
print(out.shape)

torch.Size([2, 4, 768])


In [22]:
class LLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.tok_emb = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.pos_emb = nn.Embedding(config["context_length"], config["emb_dim"])
        self.drop = nn.Dropout(config["drop_rate"])
        self.transformerBlock = nn.Sequential(
            *[Transformer(config) for _ in range(config["n_layers"])],
        )
        self.final_norm = nn.RMSNorm(config["emb_dim"])
        self.out_head = nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.drop(x)
        x = self.transformerBlock(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [23]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch.shape)

torch.Size([2, 4])


In [24]:
gpt = LLM(LLM_CONFIG)
out = gpt(batch)
print(out.shape)
print(sum(p.numel() for p in gpt.parameters()))

torch.Size([2, 4, 50257])
162971904
